# Model 10: Turkey Problem
This notebook demonstrates the 'Turkey problem' in time series forecasting, highlighting the risks of overfitting and model selection bias.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

## The Turkey Problem: Setup
The 'Turkey problem' refers to the risk of blindly trusting a model that fits historical data well but fails catastrophically in the future.

In [ ]:
# Simulate a time series with a sudden regime change
np.random.seed(42)
timesteps = np.arange(0, 500)
prices = np.concatenate([np.random.normal(100, 2, 400), np.random.normal(50, 2, 100)])
plt.figure(figsize=(10, 7))
plt.plot(timesteps, prices)
plt.title('Turkey Problem: Sudden Regime Change')
plt.xlabel('Time')
plt.ylabel('Simulated Price')
plt.grid(True)

## Train a Model on Pre-Change Data

In [ ]:
WINDOW_SIZE = 7
HORIZON = 1
def make_windows(x, window_size=7, horizon=1):
    window_step = np.expand_dims(np.arange(window_size+horizon), axis=0)
    window_indexes = window_step + np.expand_dims(np.arange(len(x)-(window_size+horizon-1)), axis=0).T
    windowed_array = x[window_indexes]
    windows = windowed_array[:, :-horizon]
    labels = windowed_array[:, -horizon:]
    return windows, labels
train_prices = prices[:400]
windows, labels = make_windows(train_prices, window_size=WINDOW_SIZE, horizon=HORIZON)
from tensorflow.keras import layers
model_turkey = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(WINDOW_SIZE,)),
    layers.Dense(HORIZON)
])
model_turkey.compile(loss='mae', optimizer=tf.keras.optimizers.Adam())
model_turkey.fit(windows, labels, epochs=100, verbose=0, batch_size=128)

## Predict After Regime Change

In [ ]:
test_prices = prices[400-WINDOW_SIZE:]
test_windows, test_labels = make_windows(test_prices, window_size=WINDOW_SIZE, horizon=HORIZON)
preds = model_turkey.predict(test_windows)
plt.figure(figsize=(10, 7))
plt.plot(np.arange(400, 500), test_labels[:, 0], label='Actual')
plt.plot(np.arange(400, 500), preds[:, 0], label='Turkey Model Prediction')
plt.title('Turkey Problem: Model Fails After Regime Change')
plt.xlabel('Time')
plt.ylabel('Simulated Price')
plt.legend()
plt.grid(True)

## Lesson: Always Validate Models on Out-of-Sample Data
The Turkey problem highlights the importance of robust validation and skepticism when forecasting time series data.